# server.py
**網路、多人遊戲與 Web 全端 › 🚀 延伸:MQTT 捷徑路線(聊天室、多人遊戲、Pico 即時儀表板)**

這是多人搶金幣遊戲的權威伺服器。權威伺服器的意思是:位置、分數都由伺服器決定,玩家只能送「我按了哪些鍵」。對照 DAY5 的 pygame_multiplayer 和 DAY6 的 web_shooter,原本五百多行要管 socket、執行緒、鎖、切封包、廣播;這裡連線和廣播交給 broker,只剩一百多行的遊戲規則,另外用 SQLite 存歷史排行榜。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-NETWEB/7/3)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!pip -q install paho-mqtt
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), 'server.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

### 說明與匯入
開頭註解把設計講得很清楚:連線、廣播、斷線偵測交給 broker;玩家只送按鍵,位置由伺服器算,所以改瀏覽器也不能瞬移;MQTT 只轉送不存資料,所以排行榜要自己存。sqlite3 是 Python 內建的小型資料庫;threading 這次只借它的鎖。

In [ ]:
# MQTT 多人遊戲 — Python「權威伺服器」(server-authoritative)
# 對照 DAY5 pygame_multiplayer/server.py、DAY6 web_shooter/server.py:
#   連線、廣播、斷線偵測都交給 MQTT broker;這支只寫「遊戲規則」
#   玩家只能送「我按了哪些鍵」,位置一律由伺服器計算 → 改瀏覽器也沒辦法瞬移
#   MQTT 只負責轉送、不存資料 → 歷史排行榜由這支存進 SQLite,再用「保留訊息」發給大家
# 安裝:pip install paho-mqtt
# 執行:python server.py [房間名稱]      (game.html 要用同一個房間)

import json
import math
import random
import sqlite3
import sys
import threading
import time

### 遊戲設定
主題前綴多了 game 這一層,和聊天分開。地圖八百乘五百;SPEED 是每秒最多走兩百二十像素,這個數字只存在伺服器,玩家改不到。TICK 是二十分之一秒,也就是每秒算二十次。超過五秒沒收到按鍵就當作離線。MAX_PLAYERS 限制最多三十人,免得有人在公開 broker 上灌一堆假玩家。

In [ ]:
import paho.mqtt.client as mqtt

# ====== 1. 設定 ======
ROOM    = sys.argv[1] if len(sys.argv) > 1 else "demo"
PREFIX  = f"pyl-academy/v1/{ROOM}/game/"
BROKER  = "broker.emqx.io"
W, H    = 800, 500         # 地圖大小
SPEED   = 220              # 每秒最多移動幾像素(伺服器說了算)
TICK    = 1 / 20           # 每秒算 20 次
TIMEOUT = 5                # 幾秒沒收到按鍵就當作離線
N_COINS = 6
MAX_PLAYERS = 30         # 公開 broker 誰都能送,限制人數免得被灌爆
COLORS  = ["#4aa8e8", "#e0283f", "#3fbf7f", "#c77dff", "#ff9e6b", "#f2ede3"]   # 不用金色,金色留給金幣

### 玩家資料與一把鎖
players 字典記錄每位玩家的名字、位置、按鍵和分數。下面的 lock 是一把鎖:收按鍵的背景執行緒會新增玩家,主迴圈會走訪玩家,兩邊同時動同一個字典就可能當掉。拿到鎖的一方先做,另一方排隊,這就是 DAY5 學過的加鎖觀念。

In [ ]:
players = {}               # 玩家ID -> {name, x, y, keys, score, color, seen}

### SQLite 排行榜
SQLite 是存在一個檔案裡的小型資料庫,伺服器重開分數還在。記錄分數用一句 SQL:新玩家就新增,舊玩家只保留最高分。注意問號:值不是直接拼進指令,而是另外交給資料庫,這能防 SQL 注入,也就是壞人把名字取成一段 SQL 指令來竄改資料庫。發布排行榜時設成保留訊息,晚進來的人也看得到。最後一行隨機放六枚金幣。

In [ ]:
lock = threading.Lock()    # 收訊執行緒和主迴圈都會碰 players → 要排隊,避免邊走訪邊新增而當掉

# ====== 排行榜:存在 SQLite(同 DAY8 專題的 sqlite3 用法),伺服器重開也還在 ======
db = sqlite3.connect(f"排行榜_{ROOM}.db")
db.execute("CREATE TABLE IF NOT EXISTS best (name TEXT PRIMARY KEY, score INTEGER)")
排行榜變了 = True


def 記錄分數(name, score):
    global 排行榜變了
    # 新玩家直接新增;舊玩家只有破紀錄才更新(參數化 ? 防 SQL 注入)
    db.execute("INSERT INTO best VALUES (?, ?) ON CONFLICT(name) DO UPDATE SET score = MAX(score, excluded.score)", (name, score))
    db.commit()
    排行榜變了 = True


def 發布排行榜(client):
    global 排行榜變了
    top = db.execute("SELECT name, score FROM best ORDER BY score DESC LIMIT 10").fetchall()
    client.publish(PREFIX + "leaderboard", json.dumps(top, ensure_ascii=False), retain=True)   # 保留:晚來的人也看得到
    排行榜變了 = False

In [ ]:
coins = [[random.randint(30, W - 30), random.randint(30, H - 30)] for _ in range(N_COINS)]

### 收到按鍵:不信任座標
玩家編號從主題最後一段取。內容先試著解析,只收上下左右四個鍵,而且值要真的是 true;名字最多十二個字。格式不對就丟掉,因為公開 broker 誰都能送東西進來。對照 web_shooter 原檔:它直接收玩家送來的 x、y,只限制在地圖內,改一下瀏覽器就能瞬移;這裡根本不讀座標。

### 加鎖登記玩家
要動 players 之前,先用 with lock 拿到鎖,離開區塊會自動還回去。第一次看到的人,如果人數已經到上限就不理他,否則放在地圖中央、零分、輪流挑顏色。最後記下他的按鍵、名字和這次出現的時間,主迴圈靠這個時間判斷他是不是離線了。

### 每一格:移動和吃金幣
先把太久沒消息的玩家踢掉,而且是先列出名單再刪,避免 ChatServer 那種一邊走訪字典一邊刪的錯誤。接著依按鍵算方向,右減左、下減上;斜走時除以長度,速度才不會變快,這正是原課程修過的斜走問題。位置乘上經過的秒數 dt,再限制在地圖內。玩家和金幣距離小於二十二就加一分、記分、金幣換位置。

In [ ]:
# ====== 2. 收到玩家按鍵:只記下「按了什麼」,不相信任何座標 ======
def 收到按鍵(client, userdata, msg):
    pid = msg.topic.split("/")[-1][:20]
    try:
        data = json.loads(msg.payload.decode("utf-8"))
        keys = {k: data.get(k) is True for k in ("up", "down", "left", "right")}   # 只收這 4 個,而且要真的是 true(送 "yes" 不算)
        name = str(data.get("name", "玩家"))[:12]
    except (ValueError, AttributeError):
        return                                      # 格式錯就丟掉,不讓壞資料弄垮伺服器
    with lock:
        p = players.get(pid)
        if p is None:                               # 第一次看到這個人 → 加入遊戲
            if len(players) >= MAX_PLAYERS:
                return
            p = players[pid] = {"name": name, "x": W / 2, "y": H / 2, "score": 0,
                                "color": COLORS[len(players) % len(COLORS)]}
            print(f"+ {name} 加入(目前 {len(players)} 人)")
        p.update(keys=keys, name=name, seen=time.time())


# ====== 3. 每一格時間:移動、吃金幣、踢掉斷線的人,再廣播狀態 ======
def 更新一格(dt):
    now = time.time()
    for pid in [k for k, p in players.items() if now - p["seen"] > TIMEOUT]:
        print(f"- {players[pid]['name']} 離線")
        del players[pid]
    for p in players.values():
        k = p["keys"]
        dx = (k["right"] - k["left"])
        dy = (k["down"] - k["up"])
        n = math.hypot(dx, dy) or 1                  # 斜走也一樣快(除以長度)
        p["x"] = min(W, max(0, p["x"] + dx / n * SPEED * dt))
        p["y"] = min(H, max(0, p["y"] + dy / n * SPEED * dt))
        for c in coins:
            if math.hypot(p["x"] - c[0], p["y"] - c[1]) < 22:
                p["score"] += 1
                記錄分數(p["name"], p["score"])
                c[0], c[1] = random.randint(30, W - 30), random.randint(30, H - 30)

### 打包狀態
把時間、地圖大小、金幣和所有玩家打包成一個 JSON 字串。座標四捨五入成整數,訊息比較短。只送畫面需要的東西,按鍵和最後出現時間這些內部資料不送,這和 pygame 版的打包玩家是同一個想法。

### 連線與伺服器遺囑
伺服器的遺囑是在 server 主題留下保留訊息 offline,伺服器一關,網頁馬上顯示「伺服器離線」。連上時訂閱 input 斜線加號,一個訂閱就收到所有玩家的按鍵;原本要 accept 迴圈加上每人一條執行緒。接著發布 online,印出提示。loop_start 讓收訊在背景跑,主執行緒專心算遊戲。

In [ ]:
def 打包狀態():
    return json.dumps({
        "t": round(time.time(), 3), "w": W, "h": H, "coins": coins,
        "players": [{"id": pid, "name": p["name"], "x": round(p["x"]), "y": round(p["y"]),
                     "score": p["score"], "color": p["color"]} for pid, p in players.items()],
    }, ensure_ascii=False)


# ====== 4. 連線:遺囑設成 offline → 這支程式關掉時,網頁會立刻顯示「伺服器離線」 ======
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="pyl-game-server-" + ROOM)
client.will_set(PREFIX + "server", "offline", retain=True)
client.on_message = 收到按鍵
client.on_connect = lambda c, u, f, rc, p: (c.subscribe(PREFIX + "input/+"),
                                             c.publish(PREFIX + "server", "online", retain=True),
                                             print(f"伺服器上線:房間「{ROOM}」,打開 game.html?room={ROOM} 開始玩"))
client.connect(BROKER, 1883, keepalive=15)

In [ ]:
client.loop_start()

### 主迴圈:固定節拍
每一圈先拿鎖,在鎖裡更新一格並打包好狀態,這時收訊執行緒要等。發布則放在鎖外,網路慢也不會卡住收按鍵。狀態不設保留,因為每秒更新二十次。排行榜有變才重發。最後睡掉剩下的時間,讓每圈約二十分之一秒,這叫固定節拍,和 pygame 版的遊戲 Tick 一樣。

### 關閉收尾
按 Ctrl 加 C 會跳到 except 印出伺服器關閉。finally 主動發布 offline,因為正常斷線不會觸發遺囑;等零點三秒讓訊息送出,再停止背景執行緒並斷線。

In [ ]:
# ====== 5. 主迴圈:固定每 1/20 秒算一次、廣播一次 ======
try:
    last = time.time()
    while True:
        now = time.time()
        with lock:
            更新一格(now - last)
            狀態 = 打包狀態()
        last = now
        client.publish(PREFIX + "state", 狀態)
        if 排行榜變了:
            發布排行榜(client)
        time.sleep(max(0, TICK - (time.time() - now)))
except KeyboardInterrupt:
    print("伺服器關閉")
finally:
    client.publish(PREFIX + "server", "offline", retain=True)
    time.sleep(0.3)
    client.loop_stop()
    client.disconnect()

## 小結
權威伺服器等於:收按鍵、加鎖保護共用資料、固定節拍計算、廣播狀態。broker 替你處理連線、廣播和斷線,你只寫規則;要存的東西交給 SQLite。下一支 game.html 是玩家端,看它怎麼只送按鍵、只畫伺服器給的畫面。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-NETWEB/7/3) 看逐句拆解、做練習。